# finops_bronze_benefits_usage

**Purpose**: Lands the hourly savings-plan-eligible PAYG usage series per subscription from the Cost Management Benefit Recommendations API, across all configured tenants.

**Domain**: finops
**Schema**: bronze

**Inputs**:
- Azure Cost Management `benefitRecommendations` API (api-version 2025-03-01), subscription scope, `$expand=properties/usage`
- ARM `subscriptions` API for per-tenant subscription discovery

**Output**: `bronze.benefits_usage`

**Parameters** (pipeline via Variable Library):
- `tenants` (string, "" = all configured; or comma-separated prefixes e.g. "a,b")
- `lookback` (string, "Last60Days"), `term` (string, "P3Y")

**Trigger**: monthly pipeline (hard requirement — the API only looks back 60 days, so a gap of >60 days between successful runs loses hourly history permanently)

---

Each run snapshots up to 60 days of hourly eligible on-demand charges per subscription. Runs **append** with a `snapshot_date` discriminator (LAKEHOUSE_TABLES.md snapshot pattern) — overlapping windows are kept deliberately, because Azure restates late-arriving usage and overlap disagreements reveal eligibility-basis changes. Silver dedupes to the latest snapshot per hour. Same-day reruns replace that day's snapshot (idempotent).

The per-subscription `usage` series sum cleanly across subscriptions; the *recommendations* do not (single-scope forgoes shared-scope diversification) — commitment decisions come from `finops_bronze_benefits_recommendations` at billing-profile scope.

> Subscription iteration and the throttling-aware ARM GET below are candidates for `finops-core` once a third notebook needs them (wheel threshold per NOTEBOOK_STANDARDS.md).

In [ ]:
%%configure -f
{
    "vCores": 4
}

In [ ]:
%pip install -q azure-identity

In [ ]:
import logging
import time
from datetime import datetime, timedelta, timezone

import polars as pl
import requests
from azure.identity import ClientSecretCredential
from deltalake import DeltaTable

logger = logging.getLogger(__name__)

In [ ]:
tenants = ""          # "" = all configured tenant prefixes; or "a" / "a,b"
lookback = "Last60Days"  # Last7Days | Last30Days | Last60Days
term = "P3Y"          # P1Y | P3Y (affects recommendation, not the usage series shape)

## 1. Configuration

Config-driven tenancy: tenant prefixes (`a`, `b`, ...) resolve credentials from the Variable Library; unconfigured prefixes are skipped; the `tenants` parameter filters.

In [ ]:
ARM = "https://management.azure.com"
API_VERSION = "2025-03-01"
TENANT_PREFIXES = ["a", "b"]

VariableLib = notebookutils.variableLibrary.getLibrary("VariableLib")
key_vault_url = VariableLib.key_vault_url
finopshub_root_path = VariableLib.finopshub_root_path
usage_table_path = f"{finopshub_root_path}/bronze/benefits_usage"

if lookback not in ("Last7Days", "Last30Days", "Last60Days"):
    raise ValueError(f"Invalid lookback: '{lookback}'")
if term not in ("P1Y", "P3Y"):
    raise ValueError(f"Invalid term: '{term}'")

def tenant_config(prefix):
    try:
        return {
            "prefix": prefix,
            "label": f"Tenant {prefix.upper()}",
            "tenant_id": getattr(VariableLib, f"{prefix}_tenant_id"),
            "client_id": getattr(VariableLib, f"{prefix}_client_id"),
            "secret_name": getattr(VariableLib, f"{prefix}_secret_name"),
        }
    except AttributeError:
        return None

requested = [p.strip() for p in tenants.split(",") if p.strip()] or TENANT_PREFIXES
tenant_configs = [c for p in requested if (c := tenant_config(p)) is not None]
if not tenant_configs:
    raise ValueError(f"No configured tenants match request '{tenants}'")

logger.info("Tenants in scope: %s | lookback=%s term=%s", [c["label"] for c in tenant_configs], lookback, term)

## 2. ARM Helpers

Plain REST rather than the SDK — the azure-mgmt-costmanagement package lags the 2025-03-01 API version. Retries honour the Cost Management throttle header (429) and Retry-After (503); follows `nextLink` pagination.

In [ ]:
def arm_session(cfg):
    credential = ClientSecretCredential(
        tenant_id=cfg["tenant_id"],
        client_id=cfg["client_id"],
        client_secret=notebookutils.credentials.getSecret(key_vault_url, cfg["secret_name"]),
    )
    token = credential.get_token(f"{ARM}/.default")
    s = requests.Session()
    s.headers["Authorization"] = f"Bearer {token.token}"
    return s

def arm_get(session, url, params=None):
    for _ in range(6):
        r = session.get(url, params=params)
        if r.status_code in (429, 503):
            wait = int(r.headers.get("x-ms-ratelimit-microsoft.consumption-retry-after",
                                     r.headers.get("Retry-After", 10)))
            logger.warning("Throttled (%d), retrying in %ds: %s", r.status_code, wait, url)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Exhausted retries (throttling): {url}")

def arm_get_paged(session, url, params=None):
    items = []
    body = arm_get(session, url, params)
    while True:
        items.extend(body.get("value", []))
        next_link = body.get("nextLink")
        if not next_link:
            return items
        body = arm_get(session, next_link)  # nextLink carries its own query string

def list_subscriptions(session):
    subs = arm_get_paged(session, f"{ARM}/subscriptions", {"api-version": "2022-12-01"})
    return [(s["subscriptionId"], s["displayName"]) for s in subs]

## 3. Fetch Hourly Usage per Subscription

Subscription-scope call with `$expand=properties/usage`. The hourly timestamps are reconstructed from `firstConsumptionDate` + index — the API reports its actual data window, which can start/end mid-day; `totalHours` must equal the length of `usage.charges` (asserted). Subscriptions with no eligible usage return no recommendations and are skipped with a log line.

In [ ]:
ingestion_ts = datetime.now(timezone.utc)
snapshot_date = ingestion_ts.date()
rows = []

for cfg in tenant_configs:
    session = arm_session(cfg)
    subscriptions = list_subscriptions(session)
    logger.info("%s: %d subscriptions", cfg["label"], len(subscriptions))

    for sub_id, sub_name in subscriptions:
        scope_url = f"{ARM}/subscriptions/{sub_id}/providers/Microsoft.CostManagement/benefitRecommendations"
        recs = arm_get_paged(session, scope_url, {
            "api-version": API_VERSION,
            "$expand": "properties/usage",
            "$filter": f"properties/lookBackPeriod eq '{lookback}' AND properties/term eq '{term}'",
        })

        if not recs:
            logger.info("  %s (%s): no eligible usage / no recommendation", sub_name, sub_id)
            continue
        if len(recs) > 1:
            logger.warning("  %s: %d recommendations returned, using first", sub_name, len(recs))

        props = recs[0]["properties"]
        charges = (props.get("usage") or {}).get("charges") or []
        if not charges:
            logger.info("  %s (%s): recommendation has no usage series", sub_name, sub_id)
            continue

        first_hour = datetime.fromisoformat(props["firstConsumptionDate"].replace("Z", "+00:00"))
        total_hours = props.get("totalHours")
        if total_hours is not None and total_hours != len(charges):
            raise ValueError(
                f"{sub_id}: totalHours {total_hours} != charges length {len(charges)} — timestamp reconstruction unsafe"
            )

        for i, charge in enumerate(charges):
            rows.append({
                "tenant_id": cfg["tenant_id"],
                "tenant_label": cfg["label"],
                "subscription_id": sub_id,
                "subscription_name": sub_name,
                "hour_utc": first_hour + timedelta(hours=i),
                "hourly_charge": float(charge),
                "currency_code": props.get("currencyCode"),
                "recommendation_scope": props.get("scope"),
                "look_back_period": props.get("lookBackPeriod"),
                "term": props.get("term"),
            })
        logger.info("  %s: %d hourly records (%s -> %s)", sub_name, len(charges),
                    props["firstConsumptionDate"], props["lastConsumptionDate"])

logger.info("Total: %d hourly records across tenants", len(rows))

## 4. Write Snapshot — `bronze.benefits_usage`

Appends this run as a new `snapshot_date`; a same-day rerun replaces that snapshot only (idempotent). Standard bronze audit columns per LAKEHOUSE_TABLES.md.

In [ ]:
if not rows:
    raise RuntimeError("No usage data retrieved from any tenant — failing loudly rather than writing an empty snapshot")

df = pl.from_dicts(rows).with_columns(
    pl.lit(ingestion_ts).alias("ingestion_timestamp"),
    pl.lit(f"benefitRecommendations api-version={API_VERSION}").alias("source_file"),
    pl.lit(snapshot_date).alias("snapshot_date"),
)

try:
    DeltaTable(usage_table_path)
    table_exists = True
except Exception:
    table_exists = False

if not table_exists:
    df.write_delta(
        usage_table_path,
        mode="overwrite",
        delta_write_options={"partition_by": ["snapshot_date"], "engine": "rust"},
    )
else:
    df.write_delta(
        usage_table_path,
        mode="overwrite",
        delta_write_options={"predicate": f"snapshot_date = '{snapshot_date}'", "engine": "rust"},
    )

logger.info("Written %d rows to bronze.benefits_usage, snapshot_date=%s", len(df), snapshot_date)

## 5. Continuity Check

Gaps in this dataset are **permanently unrecoverable** (60-day API window). Compares this snapshot's start against the previous snapshots' coverage per subscription and fails the run — loudly, for the pipeline to alert on — if a gap has opened.

In [ ]:
existing = pl.scan_delta(usage_table_path).filter(pl.col("snapshot_date") < snapshot_date)
prior_max = existing.group_by("subscription_id").agg(pl.col("hour_utc").max().alias("prior_max_hour")).collect()

if prior_max.is_empty():
    logger.info("First snapshot — no continuity to check")
else:
    new_min = df.group_by("subscription_id").agg(pl.col("hour_utc").min().alias("new_min_hour"))
    gaps = (
        prior_max.join(new_min, on="subscription_id", how="inner")
        .with_columns((pl.col("new_min_hour") - pl.col("prior_max_hour")).dt.total_hours().alias("gap_hours"))
        .filter(pl.col("gap_hours") > 1)
    )
    if gaps.is_empty():
        logger.info("Continuity OK: all %d previously-seen subscriptions overlap or abut", len(prior_max))
    else:
        display(gaps)
        raise RuntimeError(
            f"Hourly continuity broken for {len(gaps)} subscription(s) — "
            "data in the gap is unrecoverable; investigate run cadence"
        )

display(
    df.group_by(["tenant_label", "subscription_name"]).agg(
        pl.len().alias("hours"),
        pl.col("hour_utc").min().alias("from"),
        pl.col("hour_utc").max().alias("to"),
        pl.col("hourly_charge").sum().alias("total_eligible_charge"),
    ).sort(["tenant_label", "subscription_name"])
)